# Visible-State Baseline vs Raw GRPO CEO

This notebook trains and evaluates a raw GRPO CEO after the simulator exposes environment factors directly to agents: last event, recent events, market demand, competition level, economic condition, and pending effects.

Comparison target from local visible-state baseline eval, 50 episodes, seed 7:

- Survival rate: `0.94`
- Average final money: `18417.558`
- Average final users: `132.94`
- Average total reward: `-13.143`


## 1. Runtime Setup

Use a GPU runtime. A T4 works, but this run may take several hours.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/FarhanImtiaz/multiagent_startup_simulation_env.git
%cd /content/multiagent_startup_simulation_env
!git pull origin main

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt
!python -m pip install -r requirements-training.txt
!python -m pip uninstall -y torchao

In [ ]:
!python -c "import torch; print('CUDA:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')"

## 2. Recompute Visible-State Baseline

This evaluates the deterministic baseline CEO with the same visible observation fields. It does not train anything.

In [ ]:
!python evaluation.py \
  --episodes 50 \
  --horizon 30 \
  --seed 7 \
  --save-dir outputs/eval_visible_baseline

## 3. Generate Visible GRPO Dataset

This creates GRPO prompts from high-quality survivor trajectories. The prompts now include visible environment factors.

In [ ]:
!python train.py \
  --episodes 1000 \
  --horizon 30 \
  --seed 3000 \
  --output outputs/visible_grpo_trajectories.json \
  --grpo-output outputs/ceo_grpo_visible.jsonl \
  --survivors-only \
  --min-final-money 15000

In [ ]:
!wc -l outputs/ceo_grpo_visible.jsonl
!head -n 1 outputs/ceo_grpo_visible.jsonl

## 4. Train Raw GRPO CEO

This writes checkpoints to Drive so the run can survive Colab disconnects.

In [ ]:
!python train_ceo_grpo.py \
  --dataset outputs/ceo_grpo_visible.jsonl \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --output-dir /content/drive/MyDrive/ceo-grpo-visible-raw \
  --epochs 10 \
  --batch-size 4 \
  --num-generations 4 \
  --gradient-accumulation-steps 8 \
  --learning-rate 2e-7 \
  --save-steps 50 \
  --logging-steps 10 \
  --max-steps 2000 \
  --report-to tensorboard

### Resume Cell

Run this cell only after reconnecting or if the first training cell stops early. If `/content` was reset, rerun setup and dataset generation first.

In [ ]:
!python train_ceo_grpo.py \
  --dataset outputs/ceo_grpo_visible.jsonl \
  --model Qwen/Qwen2.5-0.5B-Instruct \
  --output-dir /content/drive/MyDrive/ceo-grpo-visible-raw \
  --epochs 10 \
  --batch-size 4 \
  --num-generations 4 \
  --gradient-accumulation-steps 8 \
  --learning-rate 2e-7 \
  --save-steps 50 \
  --logging-steps 10 \
  --max-steps 2000 \
  --report-to tensorboard \
  --resume-from-checkpoint latest

## 5. Evaluate Raw GRPO CEO

This bypasses the governed CEO and evaluates the adapter directly.

In [ ]:
!ls /content/drive/MyDrive/ceo-grpo-visible-raw | grep checkpoint | sort -V | tail

In [ ]:
import json
from collections import Counter
from statistics import mean

from agents import TechCoFounder, GrowthCoFounder, FinanceCoFounder, ActionProposal
from environment import StartupEnvironment
from llm_agents import HuggingFaceActionGenerator, parse_action

ADAPTER_PATH = '/content/drive/MyDrive/ceo-grpo-visible-raw/checkpoint-2000'

class RawTrainedCEO:
    name = 'CEO'

    def __init__(self):
        self.generator = HuggingFaceActionGenerator(
            base_model='Qwen/Qwen2.5-0.5B-Instruct',
            adapter_path=ADAPTER_PATH,
        )

    def choose_action(self, proposals, observation):
        proposal_lines = [
            f'- {name}: {proposal.action} | {proposal.reasoning}'
            for name, proposal in proposals.items()
        ]
        messages = [
            {
                'role': 'system',
                'content': 'You are the CEO in a startup simulator. Choose one valid action from co-founder proposals while balancing survival, recovery, and growth.',
            },
            {
                'role': 'user',
                'content': (
                    'Observed startup state:\n'
                    f'- Day: {observation["day"]}\n'
                    f'- Money: {observation["money"]}\n'
                    f'- Users: {observation["users"]}\n'
                    f'- Product quality: {observation["product_quality"]}\n'
                    f'- Team size: {observation["team_size"]}\n'
                    f'- Burn rate: {observation["burn_rate"]}\n'
                    f'- Recent user growth: {observation["recent_user_growth"]}\n'
                    f'- Last 3 growth: {observation["last_3_growth"]}\n'
                    f'- Trend direction: {observation["trend_direction"]}\n'
                    f'- Ad performance: {observation["ad_performance"]}\n'
                    f'- Runway hint: {observation["runway_hint"]}\n'
                    f'- Crisis level: {observation["crisis_level"]}\n'
                    f'- Crisis reason: {observation["crisis_reason"]}\n'
                    f'- Last event: {observation["last_event"]}\n'
                    f'- Recent events: {observation["recent_events"]}\n'
                    f'- Market demand: {observation["market_demand"]}\n'
                    f'- Competition level: {observation["competition_level"]}\n'
                    f'- Economic condition: {observation["economic_condition"]}\n'
                    f'- Pending effects: {observation["pending_effects"]}\n'
                    f'- Recent actions: {observation["recent_actions"]}\n\n'
                    'Co-founder proposals:\n'
                    f'{chr(10).join(proposal_lines)}\n\n'
                    'Respond with exactly one line in the form: Action: <action_name>'
                ),
            },
        ]
        raw = self.generator.generate_from_messages(messages)
        action = parse_action(raw, StartupEnvironment.ACTIONS) or 'do_nothing'
        return ActionProposal(action=action, reasoning=f'Raw trained CEO: {raw}')

def run_raw_episode(seed, ceo, horizon=30):
    env = StartupEnvironment(max_days=horizon, seed=seed)
    obs = env.reset()
    tech, growth, finance = TechCoFounder(), GrowthCoFounder(), FinanceCoFounder()
    total_reward = 0.0
    actions = []
    result = {'termination_reason': 'not_started'}

    for _ in range(horizon):
        proposals = {
            tech.name: tech.propose(obs),
            growth.name: growth.propose(obs),
            finance.name: finance.propose(obs),
        }
        selected = ceo.choose_action(proposals, obs)
        actions.append(selected.action)
        result = env.step(selected.action, proposals=proposals)
        total_reward += result['reward']
        obs = result['state']
        if result['done']:
            break

    final_state = env.get_debug_state()['public_state']
    return {
        'total_reward': round(total_reward, 3),
        'final_money': final_state['money'],
        'final_users': final_state['users'],
        'survived': final_state['money'] >= 0 and final_state['users'] > 0,
        'termination_reason': result.get('termination_reason'),
        'actions': actions,
    }

ceo = RawTrainedCEO()
episodes = [run_raw_episode(7 + i, ceo) for i in range(50)]

raw_summary = {
    'episodes': len(episodes),
    'average_total_reward': round(mean(e['total_reward'] for e in episodes), 3),
    'average_final_money': round(mean(e['final_money'] for e in episodes), 3),
    'average_final_users': round(mean(e['final_users'] for e in episodes), 3),
    'survival_rate': round(sum(e['survived'] for e in episodes) / len(episodes), 3),
    'termination_reasons': dict(Counter(e['termination_reason'] for e in episodes)),
    'action_counts': dict(Counter(a for e in episodes for a in e['actions'])),
}

print(json.dumps(raw_summary, indent=2))

## 6. Compare Baseline vs Raw GRPO

In [ ]:
baseline = json.load(open('outputs/eval_visible_baseline/evaluation_summary.json'))['aggregate']
comparison = {
    'visible_baseline_ceo': baseline,
    'raw_grpo_ceo': raw_summary,
    'deltas_raw_minus_baseline': {
        key: round(float(raw_summary[key]) - float(baseline[key]), 3)
        for key in ['average_total_reward', 'average_final_money', 'average_final_users', 'survival_rate']
    },
}
print(json.dumps(comparison, indent=2))